# Example 11.6 / 2: A One-Market Equilibrium

Companion notebook for **11.6 Complementarity Constraints** in the AMPL lab Documentation, section "A One-Market Equilibrium". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## A One-Market Equilibrium

Supply is $2p$ and demand is 6. The complementarity pair says price is nonnegative, excess supply is nonnegative, and a positive price requires market clearing. `expand` shows the pair with the data filled in:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param demand > 0;  # quantity demanded
param slope > 0;   # supply per unit of price

var Price;  # market price; sign set by the pair

subject to Market:  # positive price only if the market clears
    Price >= 0 complements slope * Price >= demand;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["demand"] = 6
ampl.param["slope"] = 2

In [5]:
# run.py
print(ampl.get_output("expand Market;"), end="")

subject to Market:
	0 <= Price
     complements
	6 <= 2*Price;



### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
param demand := 6;
param slope := 2;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
expand Market;

subject to Market:
	0 <= Price
     complements
	6 <= 2*Price;



### Another way in amplpy

This square system, one variable and one complementarity condition, is a complementarity
problem of the NCP kind (see Equilibrium Problems and MPCC (AMPL lab documentation, section 11.6.1)). With the same `model.mod` and `data.py`,
HiGHS returns the market-clearing price `Price = 3`, where supply $2\cdot3$ equals demand 6. On a
model this small AMPL's presolve already fixes the price before HiGHS starts:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.param["demand"] = 6
ampl.param["slope"] = 2

In [11]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.var["Price"].value(), ampl.con["Market"].slack())

solved 3.0 0.0


### Another way in amplpy

Complementarity constraints have left- and right-side suffixes such as `.Lbody`, `.Rbody`,
`.Lslack`, and `.Rslack`. For paired inequalities, `.slack` is nonnegative exactly when both sides
are feasible and is zero when complementarity holds exactly. Interpret residuals using tolerances
rather than exact floating-point equality.

The suffixes on the `Market` pair at three trial prices, with the same `model.mod` and `data.py`.
AMPL presolve would otherwise fix `Price` at 3 by itself, so `presolve` is set to 0 to evaluate
exactly the assigned values; the expressions are read with `get_value()`:

In [12]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [13]:
# data.py
ampl.param["demand"] = 6
ampl.param["slope"] = 2

In [14]:
# run.py
ampl.option["presolve"] = 0
names = ["Market.Lbody", "Market.Lslack", "Market.Rbody", "Market.Rslack", "Market.slack"]
for price in (3, 4, 0):
    ampl.var["Price"].set_value(price)
    print(price, [ampl.get_value(n) for n in names])

3 [3, 3, 6, 0, 0]
4 [4, 4, 8, 2, 2]
0 [0, 0, 0, -6, -6]


### Another way in amplpy

At 3 the pair holds exactly (`slack` 0). At 4 both sides are slack, so `slack` is 2 and
complementarity fails. At 0 the right side is violated, so `slack` is -6.

Generic names count complementarity constraints: `_nccons` in the model, and after presolve
`_sncons` (all constraints), `_snccons` (complementarity constraints), `_snlcc` and `_snnlcc` (the
linear and nonlinear ones among them). With `presolve 0` one source complementarity constraint
counts as two ordinary constraints in `_ncons`, and the solver sees one linear complementarity
constraint:

In [15]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [16]:
# data.py
ampl.param["demand"] = 6
ampl.param["slope"] = 2

In [17]:
# run.py
ampl.option["presolve"] = 0
names = ["_ncons", "_nccons", "_sncons", "_snccons", "_snlcc", "_snnlcc"]
print(dict(zip(names, (ampl.get_value(n) for n in names))))

{'_ncons': 2, '_nccons': 1, '_sncons': 1, '_snccons': 1, '_snlcc': 1, '_snnlcc': 0}
